In [1]:
import os
import numexpr
from datetime import datetime
from fpdf import FPDF

In [2]:
# Modern LangChain imports
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.prompts import ChatPromptTemplate

c:\Python\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==========================================
# 1. Define the 3 Core Tools
# ==========================================

# Tool 1: Arithmetic Solver
@tool
def calculate(expression: str) -> str:
    """
    Evaluates mathematical expressions safely. 
    Input must be a valid mathematical expression (e.g., '45 * 32' or '100 / 4').
    """
    try:
        result = numexpr.evaluate(expression)
        return str(result.item())
    except Exception as e:
        return f"Error evaluating expression: {e}"

# Tool 2: Wikipedia Extractor
wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=2000),
    description="Searches Wikipedia for factual information and summaries. Input should be a specific topic."
)

# Tool 3: PDF Document Creator
@tool
def create_pdf(content: str) -> str:
    """
    Generates a PDF document from the provided text string. 
    Input should be the final, nicely formatted text you want written into the PDF.
    """
    folder = "generated_docs"
    os.makedirs(folder, exist_ok=True)
    
    filename = f"Summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    filepath = os.path.join(folder, filename)
    
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    
    # Encode to latin-1 to avoid FPDF unicode errors with special characters
    safe_content = content.encode('latin-1', 'replace').decode('latin-1')
    
    for line in safe_content.split('\n'):
        pdf.multi_cell(0, 10, txt=line)
        
    pdf.output(filepath)
    return f"PDF Document successfully created and saved at: {filepath}"

# Bundle the tools together
tools = [calculate, wikipedia_tool, create_pdf]

In [4]:
# ==========================================
# 2. Initialize LLM and Agent Framework
# ==========================================

# Connect to your local LM Studio instance
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio", # Bypasses the OpenAI Key requirement
    temperature=0.2
)

# Define the precise routing logic based on your 3 requirements
prompt = ChatPromptTemplate.from_messages([
    (
        "system", 
        "You are an Agentic AI that processes user input and takes action based on the following rules:\n"
        "1. If the input is an arithmetic problem, use the 'calculate' tool to solve it.\n"
        "2. If the user asks for information on a topic, use the 'wikipedia' tool to extract it.\n"
        "3. If the user asks to create a PDF document about a topic, you MUST first use the 'wikipedia' "
        "tool to extract the information, format it nicely, and THEN pass that text into the 'create_pdf' tool."
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Create the agent
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [5]:
# ==========================================
# 3. Interactive User Chat Loop
# ==========================================

if __name__ == "__main__":
    print("=====================================================")
    print("Agentic AI Initialized. Type 'exit' or 'quit' to stop.")
    print("=====================================================\n")
    
    while True:
        # 1. Take dynamic input from the user
        user_input = input("\nYou: ")
        
        # 2. Provide a way to break the loop and exit the script
        if user_input.lower() in ['exit', 'quit']:
            print("Shutting down Agentic AI. Goodbye!")
            break
            
        # Skip empty inputs
        if not user_input.strip():
            continue
            
        print("\nAgent is thinking...")
        
        try:
            # 3. Pass the dynamic input to the agent
            response = agent_executor.invoke({"input": user_input})
            
            print("\n" + "="*40)
            print("FINAL OUTPUT:")
            print("="*40)
            print(response["output"])
            
        except Exception as e:
            print(f"\nAn error occurred: {e}")

Agentic AI Initialized. Type 'exit' or 'quit' to stop.


Agent is thinking...


> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'tokens in AI'}`
responded: 



Page: Large language model
Summary: A large language model (LLM) is a neural network trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can generate, summarize, translate and parse text in many contexts, and are a foundational technology behind modern chatbots. Biased or inaccurate training data can make an LLM's output less reliable. 
As of 2024, the largest and most capable LLMs are all based on transformer architectures, which, according to the 2017 paper Attention Is All You Need, can be more efficient and parallelizable than earlier statistical and recurrent neural network models. Research into other architectures, such as state space models, is ongoing.
Benchmark evaluations for LLMs attempt to measure model reasoning, factual accuracy, 